# 04-fix. §8 에러 수정 + Confidence Threshold 비교 실험

**수정 사항**:
1. `llm_agrees_bert` 컬럼의 mixed type (bool + str) 에러 수정
2. θ=0.3, 0.5, 0.7 세 가지 threshold로 성능 비교

**전략**: BERT 예측 체크포인트(`bert_predictions.parquet`)를 재활용하여 BERT 재학습 없이 진행.
threshold별로 LLM 호출 대상이 달라지므로, 각 threshold마다 LLM 분류를 별도 실행한다.

**비용**: θ=0.3에서 ~₩449, θ=0.5에서 ~₩526 추가 (θ=0.7은 기존 결과 재활용)

## 0. 환경 설정

In [1]:
# === 환경 설정 ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/5stone_experiment/1_clustering_test')

!pip install -q google-genai

import json
import time
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f'작업 디렉토리: {os.getcwd()}')

Mounted at /content/drive
작업 디렉토리: /content/drive/MyDrive/5stone_experiment/1_clustering_test


## 0-1. 경로, 파라미터, Gemini 초기화

In [2]:
# === 경로 ===
BASE_DIR = Path('/content/drive/MyDrive/5stone_experiment/1_clustering_test')
PATH_CONFIG = {
    'processed': BASE_DIR / 'data' / 'processed',
    'loop_ckpt_dir': BASE_DIR / 'checkpoints' / 'loop',
}
PATH_CONFIG['bert_predictions'] = PATH_CONFIG['loop_ckpt_dir'] / 'bert_predictions.parquet'
PATH_CONFIG['intent_descriptions'] = PATH_CONFIG['processed'] / 'intent_descriptions.parquet'

# === 비교 대상 threshold ===
THRESHOLDS = [0.3, 0.5, 0.7]

CONFIG = {
    'gemini_model': 'gemini-2.5-flash-lite',
    'gemini_temperature': 0.1,
    'gemini_max_output_tokens': 256,
    'gemini_rate_limit_delay': 0.25,
    'gemini_retry_max': 3,
    'gemini_retry_delay': 5,
    'top_k_candidates': 5,
    'null_intent_enabled': True,
}

print(f'비교 threshold: {THRESHOLDS}')
for key in ['bert_predictions', 'intent_descriptions']:
    marker = '✓' if PATH_CONFIG[key].exists() else '✗'
    print(f'  {marker} {key}')

비교 threshold: [0.3, 0.5, 0.7]
  ✓ bert_predictions
  ✓ intent_descriptions


In [3]:
# === Gemini API 초기화 ===
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def nfc(text):
    if not isinstance(text, str):
        return str(text)
    return unicodedata.normalize('NFC', text)

def call_gemini(prompt, system_instruction):
    for attempt in range(CONFIG['gemini_retry_max']):
        try:
            response = client.models.generate_content(
                model=CONFIG['gemini_model'],
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=CONFIG['gemini_temperature'],
                    max_output_tokens=CONFIG['gemini_max_output_tokens'],
                    response_mime_type='application/json',
                ),
            )
            time.sleep(CONFIG['gemini_rate_limit_delay'])
            return response.text.strip()
        except Exception as e:
            err = str(e)
            if '429' in err or 'RESOURCE_EXHAUSTED' in err:
                time.sleep(CONFIG['gemini_retry_delay'] * (attempt + 1) * 2)
            else:
                time.sleep(CONFIG['gemini_retry_delay'] * (attempt + 1))
    return ''

print('Gemini 초기화 완료')

Gemini 초기화 완료


## 1. 데이터 로드
BERT 예측 체크포인트 + Description 로드

In [4]:
# === 데이터 로드 ===

# BERT 예측 (이미 완료된 체크포인트)
bert_pred_df = pd.read_parquet(PATH_CONFIG['bert_predictions'])
print(f'BERT 예측 로드: {len(bert_pred_df):,}건')
print(f'  Top-1 정확도: {(bert_pred_df["bert_top1_label"] == bert_pred_df["intent_label"]).mean()*100:.1f}%')

# Description
desc_df = pd.read_parquet(PATH_CONFIG['intent_descriptions'])
desc_map = {}
for _, row in desc_df.iterrows():
    label = nfc(row['intent'])
    differs = row['differs_from']
    if isinstance(differs, str):
        try:
            differs = json.loads(differs)
        except json.JSONDecodeError:
            differs = []
    desc_map[label] = {'definition': row['definition'], 'differs_from': differs}
print(f'Description 매핑: {len(desc_map)}개')

BERT 예측 로드: 3,737건
  Top-1 정확도: 48.3%
Description 매핑: 906개


## 2. LLM 분류 함수 정의
04 노트북과 동일한 프롬프트 + 분류 로직. 재사용을 위해 함수화.

In [5]:
# === LLM 분류 ===

CLASSIFY_SYSTEM = """당신은 고객 상담 의도 분류 전문가입니다.

## 작업
주어진 쿼리(고객 의도 요약)를 후보 의도 중 하나로 분류하거나, 어느 것에도 해당하지 않으면 "null"로 판정하세요.

## 규칙
1. 각 후보 의도의 definition과 구별 기준(differs_from)을 참고하여 가장 적합한 의도를 선택하세요.
2. 소형 모델의 Top-1 예측이 함께 제공됩니다. 이를 참고하되, 독립적으로 판단하세요.
3. 어떤 후보 의도에도 해당하지 않으면 "null"을 선택하세요.
4. 선택한 의도와 근거를 JSON으로 출력하세요.

## 출력 형식 (JSON)
{
  "selected_intent": "행위-목적" 또는 "null",
  "reasoning": "선택 근거 한 문장",
  "agrees_with_bert": true/false
}"""


def build_classify_prompt(query, bert_top1, top_k_candidates, desc_map):
    parts = [f'쿼리: {query}', f'소형 모델 Top-1 예측: {bert_top1}', '', '후보 의도:']
    for cand in top_k_candidates:
        label = cand['label']
        prob = cand['prob']
        desc = desc_map.get(label, {})
        definition = desc.get('definition', '')
        parts.append(f'\n  [{label}] (신뢰도: {prob})')
        if definition:
            parts.append(f'    정의: {definition}')
        differs = desc.get('differs_from', [])
        if isinstance(differs, str):
            try:
                differs = json.loads(differs)
            except json.JSONDecodeError:
                differs = []
        cand_labels = {c['label'] for c in top_k_candidates}
        for d in [dd for dd in differs if dd.get('intent') in cand_labels][:2]:
            parts.append(f'    ↔ {d["intent"]}와 구별: {d["distinction"]}')
    if CONFIG['null_intent_enabled']:
        parts.append('\n  [null] (위 의도 중 어느 것에도 해당하지 않음)')
    return '\n'.join(parts)


def run_llm_for_threshold(low_conf_df, desc_map, threshold_name):
    """
    특정 threshold의 저신뢰 쿼리에 대해 LLM 분류를 실행한다.
    threshold별 체크포인트를 관리한다.
    """
    ckpt_path = PATH_CONFIG['loop_ckpt_dir'] / f'llm_predictions_t{threshold_name}.parquet'
    if ckpt_path.exists():
        print(f'  체크포인트 로드: {ckpt_path.name}')
        return pd.read_parquet(ckpt_path)

    results = []
    for idx, row in tqdm(low_conf_df.iterrows(), total=len(low_conf_df),
                         desc=f'LLM (θ={threshold_name})'):
        query = row['input_text']
        bert_top1 = row['bert_top1_label']
        top_k = json.loads(row['bert_top_k'])

        prompt = build_classify_prompt(query, bert_top1, top_k, desc_map)
        response = call_gemini(prompt, CLASSIFY_SYSTEM)

        llm_label = 'null'
        reasoning = ''
        agrees = False

        if response:
            try:
                parsed = json.loads(response)
                llm_label = nfc(parsed.get('selected_intent', 'null'))
                reasoning = parsed.get('reasoning', '')
                agrees = bool(parsed.get('agrees_with_bert', False))
            except json.JSONDecodeError:
                llm_label = 'PARSE_FAIL'

        results.append({
            'llm_label': str(llm_label),
            'llm_reasoning': str(reasoning),
            'llm_agrees_bert': bool(agrees),
        })

    result_df = low_conf_df.reset_index(drop=True).copy()
    llm_df = pd.DataFrame(results)
    result_df = pd.concat([result_df, llm_df], axis=1)

    # 타입 통일 (에러 수정 핵심)
    result_df['llm_agrees_bert'] = result_df['llm_agrees_bert'].astype(bool)
    result_df['llm_label'] = result_df['llm_label'].astype(str)
    result_df['llm_reasoning'] = result_df['llm_reasoning'].astype(str)

    result_df.to_parquet(ckpt_path, index=False)
    print(f'  저장: {ckpt_path.name}')
    return result_df


print('LLM 분류 함수 정의 완료')

LLM 분류 함수 정의 완료


## 3. 최종 결정 + 평가 함수 정의

In [6]:
# === 최종 결정 + 평가 ===

def merge_and_evaluate(high_conf_df, low_conf_with_llm, threshold, bert_pred_df):
    """
    고신뢰 + 저신뢰 통합 → 최종 분류 → 성능 평가.
    반환: metrics dict
    """
    # 고신뢰: BERT 확정
    high = high_conf_df.copy()
    high['final_label'] = high['bert_top1_label']
    high['decision_source'] = 'bert_high_conf'
    high['is_null'] = False
    high['llm_label'] = ''
    high['llm_reasoning'] = ''
    high['llm_agrees_bert'] = False

    # 저신뢰: LLM 기반 결정
    low = low_conf_with_llm.copy()
    final_labels, sources, nulls = [], [], []

    for _, row in low.iterrows():
        llm_label = str(row.get('llm_label', 'null'))
        bert_label = row['bert_top1_label']
        agrees = row.get('llm_agrees_bert', False)

        if llm_label == 'null':
            final_labels.append('NULL_INTENT')
            sources.append('llm_null')
            nulls.append(True)
        elif llm_label == 'PARSE_FAIL':
            final_labels.append(bert_label)
            sources.append('bert_fallback')
            nulls.append(False)
        elif agrees or llm_label == bert_label:
            final_labels.append(llm_label)
            sources.append('bert_llm_agree')
            nulls.append(False)
        else:
            final_labels.append(llm_label)
            sources.append('llm_override')
            nulls.append(False)

    low['final_label'] = final_labels
    low['decision_source'] = sources
    low['is_null'] = nulls

    # 타입 통일
    for col in ['llm_label', 'llm_reasoning']:
        high[col] = high[col].astype(str)
        low[col] = low[col].astype(str)
    for col in ['llm_agrees_bert', 'is_null']:
        high[col] = high[col].astype(bool)
        low[col] = low[col].astype(bool)

    common_cols = [
        'source_id', 'source', 'consulting_category', 'split',
        'intent_summary', 'intent_label', 'cluster_id', 'input_text',
        'bert_top1_label', 'bert_confidence', 'bert_top_k',
        'llm_label', 'llm_reasoning', 'llm_agrees_bert',
        'final_label', 'decision_source', 'is_null',
    ]
    h_cols = [c for c in common_cols if c in high.columns]
    l_cols = [c for c in common_cols if c in low.columns]
    combined = pd.concat([high[h_cols], low[l_cols]], ignore_index=True)

    # === 평가 ===
    metrics = {'threshold': threshold}
    total = len(combined)

    # BERT only
    bert_correct = (combined['bert_top1_label'] == combined['intent_label']).sum()
    metrics['bert_only_acc'] = round(bert_correct / total, 4)

    # 전체 (non-null)
    non_null = combined[~combined['is_null']]
    nn_correct = (non_null['final_label'] == non_null['intent_label']).sum()
    metrics['overall_acc_non_null'] = round(nn_correct / len(non_null), 4) if len(non_null) > 0 else 0

    # 전체 (null을 오답으로 처리)
    all_correct = (combined['final_label'] == combined['intent_label']).sum()
    metrics['overall_acc_strict'] = round(all_correct / total, 4)

    # 고신뢰 정확도
    if len(high) > 0:
        h_correct = (high['final_label'] == high['intent_label']).sum()
        metrics['high_conf_acc'] = round(h_correct / len(high), 4)
        metrics['high_conf_count'] = len(high)
    else:
        metrics['high_conf_acc'] = 0
        metrics['high_conf_count'] = 0

    # 저신뢰 비율
    metrics['low_conf_count'] = len(low)
    metrics['low_conf_ratio'] = round(len(low) / total, 4)

    # Null 수
    metrics['null_count'] = int(combined['is_null'].sum())

    # 결정 출처별
    for src in combined['decision_source'].unique():
        sub = combined[combined['decision_source'] == src]
        sub_nn = sub[~sub['is_null']]
        if len(sub_nn) > 0:
            src_correct = (sub_nn['final_label'] == sub_nn['intent_label']).sum()
            metrics[f'acc_{src}'] = round(src_correct / len(sub_nn), 4)
        metrics[f'count_{src}'] = len(sub)

    # LLM override 순이득
    override = non_null[non_null['decision_source'] == 'llm_override']
    if len(override) > 0:
        llm_right = (override['final_label'] == override['intent_label']).sum()
        bert_would = (override['bert_top1_label'] == override['intent_label']).sum()
        metrics['override_llm_correct'] = int(llm_right)
        metrics['override_bert_would'] = int(bert_would)
        metrics['override_net_gain'] = int(llm_right - bert_would)

    return combined, metrics


print('평가 함수 정의 완료')

평가 함수 정의 완료


## 4. Threshold별 실험 실행
θ=0.3, 0.5, 0.7 각각에 대해:
1. 고신뢰/저신뢰 분리
2. 저신뢰 → LLM 분류 (threshold별 체크포인트)
3. 통합 + 평가

θ=0.7은 기존 LLM 예측(`llm_predictions.parquet`)을 재활용할 수 있지만, 에러 수정(타입 통일)이 필요하므로 `llm_predictions_t0.7.parquet`로 새로 저장한다.

In [7]:
# === Threshold별 실험 ===

all_metrics = []
all_results = {}

for threshold in THRESHOLDS:
    t_name = str(threshold)
    print(f'\n{"="*60}')
    print(f'Threshold = {threshold}')
    print(f'{"="*60}')

    # 고/저신뢰 분리
    high = bert_pred_df[bert_pred_df['bert_confidence'] >= threshold].copy()
    low = bert_pred_df[bert_pred_df['bert_confidence'] < threshold].copy()
    print(f'  고신뢰: {len(high):,} ({len(high)/len(bert_pred_df)*100:.1f}%)')
    print(f'  저신뢰: {len(low):,} ({len(low)/len(bert_pred_df)*100:.1f}%)')

    if len(high) > 0:
        h_correct = (high['bert_top1_label'] == high['intent_label']).sum()
        print(f'  고신뢰 BERT 정확도: {h_correct}/{len(high)} = {h_correct/len(high)*100:.1f}%')

    # LLM 분류
    if len(low) > 0:
        low_with_llm = run_llm_for_threshold(low, desc_map, t_name)
    else:
        low_with_llm = low.copy()
        for col in ['llm_label', 'llm_reasoning']:
            low_with_llm[col] = ''
        low_with_llm['llm_agrees_bert'] = False

    # 통합 + 평가
    combined, metrics = merge_and_evaluate(high, low_with_llm, threshold, bert_pred_df)
    all_metrics.append(metrics)
    all_results[threshold] = combined

    print(f'\n  --- 결과 ---')
    print(f'  BERT only:       {metrics["bert_only_acc"]*100:.2f}%')
    print(f'  BERT+LLM (strict): {metrics["overall_acc_strict"]*100:.2f}%')
    print(f'  BERT+LLM (non-null): {metrics["overall_acc_non_null"]*100:.2f}%')
    print(f'  Null: {metrics["null_count"]}')
    if 'override_net_gain' in metrics:
        print(f'  LLM override 순이득: {metrics["override_net_gain"]:+d}')


Threshold = 0.3
  고신뢰: 924 (24.7%)
  저신뢰: 2,813 (75.3%)
  고신뢰 BERT 정확도: 735/924 = 79.5%
  체크포인트 로드: llm_predictions_t0.3.parquet

  --- 결과 ---
  BERT only:       48.33%
  BERT+LLM (strict): 51.19%
  BERT+LLM (non-null): 59.24%
  Null: 508
  LLM override 순이득: +244

Threshold = 0.5
  고신뢰: 439 (11.7%)
  저신뢰: 3,298 (88.3%)
  고신뢰 BERT 정확도: 399/439 = 90.9%
  체크포인트 로드: llm_predictions_t0.5.parquet

  --- 결과 ---
  BERT only:       48.33%
  BERT+LLM (strict): 50.07%
  BERT+LLM (non-null): 58.73%
  Null: 551
  LLM override 순이득: +233

Threshold = 0.7
  고신뢰: 168 (4.5%)
  저신뢰: 3,569 (95.5%)
  고신뢰 BERT 정확도: 160/168 = 95.2%
  체크포인트 로드: llm_predictions_t0.7.parquet

  --- 결과 ---
  BERT only:       48.33%
  BERT+LLM (strict): 49.56%
  BERT+LLM (non-null): 58.22%
  Null: 556
  LLM override 순이득: +215


## 5. Threshold 비교 요약

In [ ]:
# === 비교 요약 ===

def print_comparison(all_metrics):
    print(f'\n{"="*70}')
    print(f'Confidence Threshold 비교 요약')
    print(f'{"="*70}')

    header = f'{"θ":>5} {"고신뢰":>8} {"고신뢰 Acc":>10} {"LLM 호출":>8} {"Null":>6} {"Strict Acc":>10} {"Non-null Acc":>12} {"LLM 순이득":>10}'
    print(header)
    print('-' * len(header))

    for m in all_metrics:
        t = m['threshold']
        high_n = m['high_conf_count']
        high_acc = m.get('high_conf_acc', 0) * 100
        low_n = m['low_conf_count']
        null_n = m['null_count']
        strict = m['overall_acc_strict'] * 100
        non_null = m['overall_acc_non_null'] * 100
        gain = m.get('override_net_gain', 0)

        print(f'{t:>5} {high_n:>8,} {high_acc:>9.1f}% {low_n:>8,} {null_n:>6} {strict:>9.2f}% {non_null:>11.2f}% {gain:>+10d}')

    # 최적 threshold 추천
    best_strict = max(all_metrics, key=lambda m: m['overall_acc_strict'])
    best_nn = max(all_metrics, key=lambda m: m['overall_acc_non_null'])

    print(f'\n최적 (Strict):   θ={best_strict["threshold"]} ({best_strict["overall_acc_strict"]*100:.2f}%)')
    print(f'최적 (Non-null): θ={best_nn["threshold"]} ({best_nn["overall_acc_non_null"]*100:.2f}%)')

    # BERT only baseline 참조
    bert_base = all_metrics[0]['bert_only_acc'] * 100
    print(f'\nBERT only baseline: {bert_base:.2f}%')
    for m in all_metrics:
        imp = m['overall_acc_strict'] * 100 - bert_base
        print(f'  θ={m["threshold"]}: {imp:+.2f}%p 개선 (strict)')


print_comparison(all_metrics)


Confidence Threshold 비교 요약
    θ      고신뢰    고신뢰 Acc   LLM 호출   Null Strict Acc Non-null Acc    LLM 순이득
----------------------------------------------------------------------------
  0.3      924      79.5%    2,813    508     51.19%       59.24%       +244
  0.5      439      90.9%    3,298    551     50.07%       58.73%       +233
  0.7      168      95.2%    3,569    556     49.56%       58.22%       +215

최적 (Strict):   θ=0.3 (51.19%)
최적 (Non-null): θ=0.3 (59.24%)

BERT only baseline: 48.33%
  θ=0.3: +2.86%p 개선 (strict)
  θ=0.5: +1.74%p 개선 (strict)
  θ=0.7: +1.23%p 개선 (strict)


## 6. 최적 Threshold 결과 저장

In [ ]:
# === 최적 결과 저장 ===

def save_best_results(all_metrics, all_results):
    # Strict 기준 최적 선택
    best = max(all_metrics, key=lambda m: m['overall_acc_strict'])
    best_t = best['threshold']
    best_df = all_results[best_t]

    print(f'최적 threshold: {best_t}')

    # 타입 통일 (에러 방지)
    for col in best_df.columns:
        if best_df[col].dtype == object:
            best_df[col] = best_df[col].astype(str)
    best_df['is_null'] = best_df['is_null'].astype(bool)
    best_df['llm_agrees_bert'] = best_df['llm_agrees_bert'].astype(str)  # 안전하게 str로

    # 분류 결과 저장
    output_path = PATH_CONFIG['processed'] / 'loop_results.parquet'
    best_df.to_parquet(output_path, index=False)
    print(f'분류 결과 저장: {output_path}')

    # 전체 메트릭 저장
    eval_path = PATH_CONFIG['processed'] / 'loop_evaluation.json'
    eval_data = {
        'best_threshold': best_t,
        'all_thresholds': all_metrics,
    }
    with open(eval_path, 'w', encoding='utf-8') as f:
        json.dump(eval_data, f, ensure_ascii=False, indent=2, default=str)
    print(f'평가 메트릭 저장: {eval_path}')

    # 비용 정산
    total_llm_calls = sum(m['low_conf_count'] for m in all_metrics)
    est_cost = total_llm_calls * 900 / 1e6 * 0.10 + total_llm_calls * 50 / 1e6 * 0.40
    cost_krw = est_cost * 1450
    total_spent = 5200 + 1000 + 160 + cost_krw
    print(f'\n=== 비용 ===')
    print(f'  이번 실험 LLM 호출: {total_llm_calls:,} (3 threshold 합산)')
    print(f'  추정 비용: ₩{cost_krw:,.0f}')
    print(f'  누적 총 비용: ~₩{total_spent:,.0f}')
    print(f'  잔여 예산: ~₩{50000-total_spent:,.0f}')


save_best_results(all_metrics, all_results)

최적 threshold: 0.3
분류 결과 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/loop_results.parquet
평가 메트릭 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/loop_evaluation.json

=== 비용 ===
  이번 실험 LLM 호출: 9,680 (3 threshold 합산)
  추정 비용: ₩1,544
  누적 총 비용: ~₩7,904
  잔여 예산: ~₩42,096


## 7. BERT PM 로드 + Top-10 추론
저장된 BERT PM 체크포인트를 로드하여 Top-10 후보 + confidence를 새로 산출한다.

In [8]:
# === GPU 확인 + 추가 라이브러리 ===
!pip install -q transformers accelerate

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

TOP_K_NEW = 10
THRESHOLD_FIXED = 0.3
print(f'Top-K: {TOP_K_NEW}, θ: {THRESHOLD_FIXED}')

Device: cuda (Tesla T4)
Top-K: 10, θ: 0.3


In [9]:
# === BERT PM 로드 ===

bert_pm_dir = PATH_CONFIG['loop_ckpt_dir'] / 'bert_pm'

tokenizer = AutoTokenizer.from_pretrained(str(bert_pm_dir))
bert_model = AutoModelForSequenceClassification.from_pretrained(str(bert_pm_dir))
bert_model.to(device)
bert_model.eval()

# id2label 재구축
intentset_df = pd.read_parquet(PATH_CONFIG['processed'] / 'initial_intentset.parquet')
intentset_df['intent_label'] = intentset_df['intent_label'].apply(nfc)
assigned = intentset_df[intentset_df['cluster_id'] >= 0]
unique_labels = sorted(assigned['intent_label'].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print(f'BERT PM 로드 완료: {len(id2label)} 클래스')

Loading weights:   0%|          | 0/201 [00:03<?, ?it/s]

BERT PM 로드 완료: 907 클래스


In [10]:
# === Top-10 추론 ===

def bert_inference_topk(model, tokenizer, df, id2label, top_k=10):
    """
    BERT PM으로 Top-K 후보를 산출한다.
    체크포인트: bert_predictions_k{top_k}.parquet
    """
    ckpt_path = PATH_CONFIG['loop_ckpt_dir'] / f'bert_predictions_k{top_k}.parquet'
    if ckpt_path.exists():
        print(f'체크포인트 로드: {ckpt_path.name}')
        return pd.read_parquet(ckpt_path)

    model.eval()
    texts = df['input_text'].tolist()
    all_results = []
    batch_size = 64

    for start in tqdm(range(0, len(texts), batch_size), desc=f'BERT Top-{top_k} 추론'):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(
            batch_texts, truncation=True, padding='max_length',
            max_length=128, return_tensors='pt',
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        topk_probs, topk_ids = torch.topk(probs, k=min(top_k, probs.shape[1]), dim=-1)

        for i in range(len(batch_texts)):
            top1_id = topk_ids[i][0].item()
            top1_conf = topk_probs[i][0].item()

            top_k_list = []
            for j in range(min(top_k, topk_ids.shape[1])):
                cid = topk_ids[i][j].item()
                prob = topk_probs[i][j].item()
                top_k_list.append({
                    'label_id': cid,
                    'label': id2label.get(cid, 'UNKNOWN'),
                    'prob': round(prob, 4),
                })

            all_results.append({
                'bert_top1_id': top1_id,
                'bert_top1_label': id2label.get(top1_id, 'UNKNOWN'),
                'bert_confidence': round(top1_conf, 4),
                'bert_top_k': json.dumps(top_k_list, ensure_ascii=False),
            })

    # 원본 test 데이터와 결합
    # bert_pred_df에서 원본 컬럼만 가져옴
    orig_cols = ['source_id', 'source', 'consulting_category', 'split',
                 'intent_summary', 'intent_label', 'cluster_id', 'input_text']
    orig_cols = [c for c in orig_cols if c in df.columns]
    result_df = df[orig_cols].reset_index(drop=True).copy()
    pred_df = pd.DataFrame(all_results)
    result_df = pd.concat([result_df, pred_df], axis=1)

    result_df.to_parquet(ckpt_path, index=False)
    print(f'저장: {ckpt_path.name}')
    return result_df


bert_pred_k10 = bert_inference_topk(bert_model, tokenizer, bert_pred_df, id2label, top_k=TOP_K_NEW)

# Top-1 정확도 (K=10이어도 Top-1은 동일해야 함)
correct = (bert_pred_k10['bert_top1_label'] == bert_pred_k10['intent_label']).sum()
print(f'\nBERT Top-1 정확도: {correct}/{len(bert_pred_k10)} = {correct/len(bert_pred_k10)*100:.1f}%')

# Top-10 recall (정답이 Top-10 안에 있는 비율)
topk_recall = 0
for _, row in bert_pred_k10.iterrows():
    top_k = json.loads(row['bert_top_k'])
    top_k_labels = {c['label'] for c in top_k}
    if row['intent_label'] in top_k_labels:
        topk_recall += 1
print(f'Top-{TOP_K_NEW} Recall: {topk_recall}/{len(bert_pred_k10)} = {topk_recall/len(bert_pred_k10)*100:.1f}%')

# 비교: Top-5 recall
top5_recall = 0
for _, row in bert_pred_df.iterrows():
    top_k = json.loads(row['bert_top_k'])
    top_k_labels = {c['label'] for c in top_k}
    if row['intent_label'] in top_k_labels:
        top5_recall += 1
print(f'Top-5 Recall:  {top5_recall}/{len(bert_pred_df)} = {top5_recall/len(bert_pred_df)*100:.1f}%')
print(f'Recall 개선:   +{(topk_recall - top5_recall)/len(bert_pred_df)*100:.1f}%p')

체크포인트 로드: bert_predictions_k10.parquet

BERT Top-1 정확도: 1805/3737 = 48.3%
Top-10 Recall: 3159/3737 = 84.5%
Top-5 Recall:  2864/3737 = 76.6%
Recall 개선:   +7.9%p


## 8. Top-10 + θ=0.3 LLM 분류 실행

In [11]:
# === Top-10, θ=0.3 LLM 분류 ===

# 고/저신뢰 분리 (θ=0.3)
high_k10 = bert_pred_k10[bert_pred_k10['bert_confidence'] >= THRESHOLD_FIXED].copy()
low_k10 = bert_pred_k10[bert_pred_k10['bert_confidence'] < THRESHOLD_FIXED].copy()

print(f'θ={THRESHOLD_FIXED}, K={TOP_K_NEW}')
print(f'  고신뢰: {len(high_k10):,} ({len(high_k10)/len(bert_pred_k10)*100:.1f}%)')
print(f'  저신뢰: {len(low_k10):,} ({len(low_k10)/len(bert_pred_k10)*100:.1f}%)')

if len(high_k10) > 0:
    h_correct = (high_k10['bert_top1_label'] == high_k10['intent_label']).sum()
    print(f'  고신뢰 BERT 정확도: {h_correct}/{len(high_k10)} = {h_correct/len(high_k10)*100:.1f}%')

# LLM 분류 (체크포인트: llm_predictions_k10_t0.3.parquet)
def run_llm_k10(low_df, desc_map):
    ckpt_path = PATH_CONFIG['loop_ckpt_dir'] / f'llm_predictions_k{TOP_K_NEW}_t{THRESHOLD_FIXED}.parquet'
    if ckpt_path.exists():
        print(f'  체크포인트 로드: {ckpt_path.name}')
        return pd.read_parquet(ckpt_path)

    results = []
    for idx, row in tqdm(low_df.iterrows(), total=len(low_df), desc=f'LLM K={TOP_K_NEW}'):
        query = row['input_text']
        bert_top1 = row['bert_top1_label']
        top_k = json.loads(row['bert_top_k'])  # 이제 10개

        prompt = build_classify_prompt(query, bert_top1, top_k, desc_map)
        response = call_gemini(prompt, CLASSIFY_SYSTEM)

        llm_label = 'null'
        reasoning = ''
        agrees = False

        if response:
            try:
                parsed = json.loads(response)
                llm_label = nfc(parsed.get('selected_intent', 'null'))
                reasoning = parsed.get('reasoning', '')
                agrees = bool(parsed.get('agrees_with_bert', False))
            except json.JSONDecodeError:
                llm_label = 'PARSE_FAIL'

        results.append({
            'llm_label': str(llm_label),
            'llm_reasoning': str(reasoning),
            'llm_agrees_bert': bool(agrees),
        })

    result_df = low_df.reset_index(drop=True).copy()
    llm_df = pd.DataFrame(results)
    result_df = pd.concat([result_df, llm_df], axis=1)
    result_df['llm_agrees_bert'] = result_df['llm_agrees_bert'].astype(bool)
    result_df['llm_label'] = result_df['llm_label'].astype(str)
    result_df['llm_reasoning'] = result_df['llm_reasoning'].astype(str)

    result_df.to_parquet(ckpt_path, index=False)
    print(f'  저장: {ckpt_path.name}')
    return result_df


low_k10_with_llm = run_llm_k10(low_k10, desc_map)

θ=0.3, K=10
  고신뢰: 924 (24.7%)
  저신뢰: 2,813 (75.3%)
  고신뢰 BERT 정확도: 735/924 = 79.5%
  체크포인트 로드: llm_predictions_k10_t0.3.parquet


## 9. Top-10 결과 평가 + Top-5와 비교

In [12]:
# === Top-10 평가 ===

# merge_and_evaluate는 §3에서 이미 정의됨
combined_k10, metrics_k10 = merge_and_evaluate(
    high_k10, low_k10_with_llm, THRESHOLD_FIXED, bert_pred_k10,
)

# Top-5 (θ=0.3) 결과 가져오기
metrics_k5 = all_metrics[0]  # θ=0.3이 첫 번째

# 비교 테이블
print(f'\n{"="*70}')
print(f'Top-K 비교 (θ=0.3 고정)')
print(f'{"="*70}')
print(f'{"":>8} {"Top-5":>12} {"Top-10":>12} {"차이":>10}')
print(f'{"-"*42}')

comparisons = [
    ('Strict Acc', metrics_k5['overall_acc_strict'], metrics_k10['overall_acc_strict']),
    ('Non-null Acc', metrics_k5['overall_acc_non_null'], metrics_k10['overall_acc_non_null']),
    ('Null 건수', metrics_k5['null_count'], metrics_k10['null_count']),
    ('LLM 순이득', metrics_k5.get('override_net_gain', 0), metrics_k10.get('override_net_gain', 0)),
]

for name, v5, v10 in comparisons:
    if 'Acc' in name:
        diff = (v10 - v5) * 100
        print(f'{name:>12} {v5*100:>11.2f}% {v10*100:>11.2f}% {diff:>+9.2f}%p')
    else:
        diff = v10 - v5
        print(f'{name:>12} {v5:>12} {v10:>12} {diff:>+10d}')

print(f'\nBERT only baseline: {metrics_k5["bert_only_acc"]*100:.2f}%')


Top-K 비교 (θ=0.3 고정)
                Top-5       Top-10         차이
------------------------------------------
  Strict Acc       51.19%       52.34%     +1.15%p
Non-null Acc       59.24%       58.28%     -0.96%p
     Null 건수          508          381       -127
     LLM 순이득          244          257        +13

BERT only baseline: 48.33%


## 10. 최종 결과 저장 (Top-K 비교 포함)

In [13]:
# === 최종 저장 ===

# 더 나은 결과를 최종 결과로 저장
best_strict_k5 = metrics_k5['overall_acc_strict']
best_strict_k10 = metrics_k10['overall_acc_strict']

if best_strict_k10 >= best_strict_k5:
    best_k = 10
    best_df = combined_k10
    best_metrics = metrics_k10
else:
    best_k = 5
    best_df = all_results[THRESHOLD_FIXED]
    best_metrics = metrics_k5

print(f'최적 설정: Top-K={best_k}, θ={THRESHOLD_FIXED}')
print(f'Strict Acc: {best_metrics["overall_acc_strict"]*100:.2f}%')

# 타입 통일 후 저장
for col in best_df.columns:
    if best_df[col].dtype == object:
        best_df[col] = best_df[col].astype(str)
best_df['is_null'] = best_df['is_null'].astype(bool)

output_path = PATH_CONFIG['processed'] / 'loop_results.parquet'
best_df.to_parquet(output_path, index=False)
print(f'분류 결과 저장: {output_path}')

# 전체 메트릭 저장 (Top-K 비교 포함)
eval_path = PATH_CONFIG['processed'] / 'loop_evaluation.json'
eval_data = {
    'best_config': {'top_k': best_k, 'threshold': THRESHOLD_FIXED},
    'top_k_5_t0.3': metrics_k5,
    'top_k_10_t0.3': metrics_k10,
    'threshold_comparison': all_metrics,
}
with open(eval_path, 'w', encoding='utf-8') as f:
    json.dump(eval_data, f, ensure_ascii=False, indent=2, default=str)
print(f'평가 메트릭 저장: {eval_path}')

# 비용 정산
total_llm = 9680 + len(low_k10)  # 기존 3 threshold + K=10 실험
est_cost = total_llm * 1000 / 1e6 * 0.10 + total_llm * 50 / 1e6 * 0.40
cost_krw = est_cost * 1450
total_spent = 5200 + 1000 + 160 + cost_krw
print(f'\n=== 비용 ===')
print(f'  총 LLM 호출 (모든 실험 합산): {total_llm:,}')
print(f'  추정 비용: ₩{cost_krw:,.0f}')
print(f'  누적: ~₩{total_spent:,.0f} / ₩50,000')
print(f'  잔여: ~₩{50000-total_spent:,.0f}')

최적 설정: Top-K=10, θ=0.3
Strict Acc: 52.34%
분류 결과 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/loop_results.parquet
평가 메트릭 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/loop_evaluation.json

=== 비용 ===
  총 LLM 호출 (모든 실험 합산): 12,493
  추정 비용: ₩2,174
  누적: ~₩8,534 / ₩50,000
  잔여: ~₩41,466


---
# §11~§14: 도메인 필터링 실험 (K=10, θ=0.3)

BERT PM은 그대로 유지하되, 추론 시 `source` 필드를 기반으로 해당 도메인의 클러스터만 Top-K 후보로 제한한다.

**원리**: 907 클래스 전체 softmax → 도메인별 logit 마스킹 → 후보 공간 축소 → confidence 향상 + 도메인 간 혼동 제거

**BERT 재학습 불필요** — 동일 모델, 추론 로직만 변경


## 11. 도메인-클러스터 매핑 구축
IntentSet에서 각 도메인(source)에 속하는 클러스터 ID와 label_id를 매핑한다.

In [14]:
# === 도메인-클러스터 매핑 ===

intentset_df = pd.read_parquet(PATH_CONFIG['processed'] / 'initial_intentset.parquet')
intentset_df['intent_label'] = intentset_df['intent_label'].apply(nfc)
intentset_df['source'] = intentset_df['source'].apply(nfc)
assigned = intentset_df[intentset_df['cluster_id'] >= 0]

# label2id 재구축
unique_labels = sorted(assigned['intent_label'].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

# 도메인별 label_id 집합
domain_label_ids = {}
for src in sorted(assigned['source'].unique()):
    sub = assigned[assigned['source'] == src]
    domain_labels = sub['intent_label'].unique()
    domain_ids = sorted([label2id[lbl] for lbl in domain_labels if lbl in label2id])
    domain_label_ids[src] = domain_ids

# 크로스 도메인 클러스터 확인 (여러 도메인에 걸친 클러스터)
label_domains = {}
for src, ids in domain_label_ids.items():
    for lid in ids:
        label_domains.setdefault(lid, set()).add(src)

cross_domain = {lid: doms for lid, doms in label_domains.items() if len(doms) > 1}

print('=== 도메인별 클러스터 수 ===')
for src, ids in domain_label_ids.items():
    print(f'  {src}: {len(ids)} 클래스')
print(f'\n전체: {len(label2id)} 클래스')
print(f'크로스 도메인 클러스터: {len(cross_domain)}개')
if cross_domain:
    for lid, doms in list(cross_domain.items())[:5]:
        print(f'  {id2label[lid]}: {doms}')

=== 도메인별 클러스터 수 ===
  액티벤처: 243 클래스
  엘지유플러스: 579 클래스
  하나카드: 608 클래스

전체: 907 클래스
크로스 도메인 클러스터: 437개
  문의-가맹점정보: {'액티벤처', '하나카드', '엘지유플러스'}
  문의-가상계좌및결제금액: {'액티벤처', '하나카드', '엘지유플러스'}
  문의-가족결합할인: {'액티벤처', '엘지유플러스'}
  문의-객실예약: {'액티벤처', '엘지유플러스'}
  문의-결제금액확인: {'액티벤처', '하나카드', '엘지유플러스'}


## 12. 도메인 필터링 Top-K 추론
BERT PM의 전체 logit에서 해당 도메인의 label_id만 추출하여 softmax → Top-K를 산출한다.

In [15]:
# === 도메인 필터링 추론 ===

def bert_inference_domain_filtered(
    model, tokenizer, test_df, id2label, domain_label_ids, top_k=10,
):
    """
    BERT PM 추론 시 source별로 해당 도메인 클러스터만 후보로 제한한다.
    체크포인트: bert_predictions_domain_k{top_k}.parquet
    """
    ckpt_path = PATH_CONFIG['loop_ckpt_dir'] / f'bert_predictions_domain_k{top_k}.parquet'
    if ckpt_path.exists():
        print(f'체크포인트 로드: {ckpt_path.name}')
        return pd.read_parquet(ckpt_path)

    model.eval()
    texts = test_df['input_text'].tolist()
    sources = test_df['source'].tolist()
    all_results = []
    batch_size = 64

    for start in tqdm(range(0, len(texts), batch_size), desc=f'도메인 필터 Top-{top_k}'):
        batch_texts = texts[start:start + batch_size]
        batch_sources = sources[start:start + batch_size]

        inputs = tokenizer(
            batch_texts, truncation=True, padding='max_length',
            max_length=128, return_tensors='pt',
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits  # (batch, 907)

        for i in range(len(batch_texts)):
            src = batch_sources[i]
            valid_ids = domain_label_ids.get(src, list(range(logits.shape[1])))

            # 도메인 마스킹: 해당 도메인 label_id의 logit만 추출
            valid_ids_tensor = torch.tensor(valid_ids, device=device)
            domain_logits = logits[i][valid_ids_tensor]  # (n_domain_classes,)
            domain_probs = F.softmax(domain_logits, dim=-1)

            # Top-K (도메인 내에서)
            k = min(top_k, len(valid_ids))
            topk_probs, topk_local_ids = torch.topk(domain_probs, k=k)

            # local index → global label_id 변환
            top1_global_id = valid_ids[topk_local_ids[0].item()]
            top1_conf = topk_probs[0].item()

            top_k_list = []
            for j in range(k):
                global_id = valid_ids[topk_local_ids[j].item()]
                prob = topk_probs[j].item()
                top_k_list.append({
                    'label_id': global_id,
                    'label': id2label.get(global_id, 'UNKNOWN'),
                    'prob': round(prob, 4),
                })

            all_results.append({
                'bert_top1_id': top1_global_id,
                'bert_top1_label': id2label.get(top1_global_id, 'UNKNOWN'),
                'bert_confidence': round(top1_conf, 4),
                'bert_top_k': json.dumps(top_k_list, ensure_ascii=False),
            })

    orig_cols = ['source_id', 'source', 'consulting_category', 'split',
                 'intent_summary', 'intent_label', 'cluster_id', 'input_text']
    orig_cols = [c for c in orig_cols if c in test_df.columns]
    result_df = test_df[orig_cols].reset_index(drop=True).copy()
    pred_df = pd.DataFrame(all_results)
    result_df = pd.concat([result_df, pred_df], axis=1)

    result_df.to_parquet(ckpt_path, index=False)
    print(f'저장: {ckpt_path.name}')
    return result_df


bert_pred_domain = bert_inference_domain_filtered(
    bert_model, tokenizer, bert_pred_df, id2label, domain_label_ids, top_k=TOP_K_NEW,
)

# 기본 통계
correct = (bert_pred_domain['bert_top1_label'] == bert_pred_domain['intent_label']).sum()
print(f'\n=== 도메인 필터링 BERT Top-1 정확도 ===')
print(f'  필터링 전: {(bert_pred_df["bert_top1_label"] == bert_pred_df["intent_label"]).sum()}/{len(bert_pred_df)} = {(bert_pred_df["bert_top1_label"] == bert_pred_df["intent_label"]).mean()*100:.1f}%')
print(f'  필터링 후: {correct}/{len(bert_pred_domain)} = {correct/len(bert_pred_domain)*100:.1f}%')

# Confidence 분포 비교
print(f'\n=== Confidence 분포 비교 ===')
for name, df_ref in [('필터링 전', bert_pred_df), ('필터링 후', bert_pred_domain)]:
    confs = df_ref['bert_confidence']
    print(f'  {name}: mean={confs.mean():.3f}, median={confs.median():.3f}, ≥0.3={int((confs>=0.3).sum())} ({(confs>=0.3).mean()*100:.1f}%)')

# Top-10 Recall
topk_recall = 0
for _, row in bert_pred_domain.iterrows():
    top_k = json.loads(row['bert_top_k'])
    if row['intent_label'] in {c['label'] for c in top_k}:
        topk_recall += 1
print(f'\nTop-10 Recall (도메인 필터): {topk_recall}/{len(bert_pred_domain)} = {topk_recall/len(bert_pred_domain)*100:.1f}%')

# 도메인별 Top-1 정확도
print(f'\n=== 도메인별 Top-1 정확도 ===')
for src in sorted(bert_pred_domain['source'].unique()):
    sub = bert_pred_domain[bert_pred_domain['source'] == src]
    src_correct = (sub['bert_top1_label'] == sub['intent_label']).sum()
    print(f'  {src}: {src_correct}/{len(sub)} = {src_correct/len(sub)*100:.1f}%')

도메인 필터 Top-10:   0%|          | 0/59 [00:00<?, ?it/s]

저장: bert_predictions_domain_k10.parquet

=== 도메인 필터링 BERT Top-1 정확도 ===
  필터링 전: 1806/3737 = 48.3%
  필터링 후: 1812/3737 = 48.5%

=== Confidence 분포 비교 ===
  필터링 전: mean=0.211, median=0.131, ≥0.3=924 (24.7%)
  필터링 후: mean=0.248, median=0.164, ≥0.3=1136 (30.4%)

Top-10 Recall (도메인 필터): 3173/3737 = 84.9%

=== 도메인별 Top-1 정확도 ===
  액티벤처: 185/417 = 44.4%
  엘지유플러스: 553/1216 = 45.5%
  하나카드: 1074/2104 = 51.0%


## 13. 도메인 필터링 + θ=0.3 + K=10 LLM 분류

In [16]:
# === 도메인 필터링 LLM 분류 ===

high_domain = bert_pred_domain[bert_pred_domain['bert_confidence'] >= THRESHOLD_FIXED].copy()
low_domain = bert_pred_domain[bert_pred_domain['bert_confidence'] < THRESHOLD_FIXED].copy()

print(f'도메인 필터 + θ={THRESHOLD_FIXED}, K={TOP_K_NEW}')
print(f'  고신뢰: {len(high_domain):,} ({len(high_domain)/len(bert_pred_domain)*100:.1f}%)')
print(f'  저신뢰: {len(low_domain):,} ({len(low_domain)/len(bert_pred_domain)*100:.1f}%)')

if len(high_domain) > 0:
    h_correct = (high_domain['bert_top1_label'] == high_domain['intent_label']).sum()
    print(f'  고신뢰 BERT 정확도: {h_correct}/{len(high_domain)} = {h_correct/len(high_domain)*100:.1f}%')

# LLM 분류
def run_llm_domain(low_df, desc_map):
    ckpt_path = PATH_CONFIG['loop_ckpt_dir'] / f'llm_predictions_domain_k{TOP_K_NEW}_t{THRESHOLD_FIXED}.parquet'
    if ckpt_path.exists():
        print(f'  체크포인트 로드: {ckpt_path.name}')
        return pd.read_parquet(ckpt_path)

    results = []
    for idx, row in tqdm(low_df.iterrows(), total=len(low_df), desc='LLM (도메인 필터)'):
        query = row['input_text']
        bert_top1 = row['bert_top1_label']
        top_k = json.loads(row['bert_top_k'])

        prompt = build_classify_prompt(query, bert_top1, top_k, desc_map)
        response = call_gemini(prompt, CLASSIFY_SYSTEM)

        llm_label = 'null'
        reasoning = ''
        agrees = False

        if response:
            try:
                parsed = json.loads(response)
                llm_label = nfc(parsed.get('selected_intent', 'null'))
                reasoning = parsed.get('reasoning', '')
                agrees = bool(parsed.get('agrees_with_bert', False))
            except json.JSONDecodeError:
                llm_label = 'PARSE_FAIL'

        results.append({
            'llm_label': str(llm_label),
            'llm_reasoning': str(reasoning),
            'llm_agrees_bert': bool(agrees),
        })

    result_df = low_df.reset_index(drop=True).copy()
    llm_df = pd.DataFrame(results)
    result_df = pd.concat([result_df, llm_df], axis=1)
    result_df['llm_agrees_bert'] = result_df['llm_agrees_bert'].astype(bool)
    result_df['llm_label'] = result_df['llm_label'].astype(str)
    result_df['llm_reasoning'] = result_df['llm_reasoning'].astype(str)

    result_df.to_parquet(ckpt_path, index=False)
    print(f'  저장: {ckpt_path.name}')
    return result_df


low_domain_with_llm = run_llm_domain(low_domain, desc_map)

도메인 필터 + θ=0.3, K=10
  고신뢰: 1,136 (30.4%)
  저신뢰: 2,601 (69.6%)
  고신뢰 BERT 정확도: 872/1136 = 76.8%


LLM (도메인 필터):   0%|          | 0/2601 [00:00<?, ?it/s]

  저장: llm_predictions_domain_k10_t0.3.parquet


## 14. 도메인 필터링 결과 평가 + 전체 비교

In [17]:
# === 평가 + 전체 비교 ===

combined_domain, metrics_domain = merge_and_evaluate(
    high_domain, low_domain_with_llm, THRESHOLD_FIXED, bert_pred_domain,
)

# 전체 실험 비교
print(f'\n{"="*75}')
print(f'전체 실험 비교 (θ=0.3 고정)')
print(f'{"="*75}')
print(f'{"설정":>20} {"BERT Top-1":>10} {"Strict":>10} {"Non-null":>10} {"Null":>6} {"고신뢰":>8} {"고신뢰Acc":>9}')
print(f'{"-"*75}')

configs = [
    ('K=5, 필터 없음', metrics_k5),
    ('K=10, 필터 없음', metrics_k10),
    ('K=10, 도메인 필터', metrics_domain),
]

for name, m in configs:
    bert_acc = m['bert_only_acc'] * 100
    strict = m['overall_acc_strict'] * 100
    nn = m['overall_acc_non_null'] * 100
    null_n = m['null_count']
    high_n = m['high_conf_count']
    high_acc = m.get('high_conf_acc', 0) * 100
    print(f'{name:>20} {bert_acc:>9.1f}% {strict:>9.2f}% {nn:>9.2f}% {null_n:>6} {high_n:>8} {high_acc:>8.1f}%')

print(f'\nBERT only baseline: 48.33%')
best_strict = max(configs, key=lambda x: x[1]['overall_acc_strict'])
print(f'최적 설정: {best_strict[0]} → Strict {best_strict[1]["overall_acc_strict"]*100:.2f}%')
print(f'BERT baseline 대비 개선: +{(best_strict[1]["overall_acc_strict"] - 0.4833)*100:.2f}%p')


전체 실험 비교 (θ=0.3 고정)
                  설정 BERT Top-1     Strict   Non-null   Null      고신뢰    고신뢰Acc
---------------------------------------------------------------------------
          K=5, 필터 없음      48.3%     51.19%     59.24%    508      924     79.5%
         K=10, 필터 없음      48.3%     52.34%     58.28%    381      924     79.5%
        K=10, 도메인 필터      48.5%     52.61%     58.27%    363     1136     76.8%

BERT only baseline: 48.33%
최적 설정: K=10, 도메인 필터 → Strict 52.61%
BERT baseline 대비 개선: +4.28%p


In [18]:
# === 최종 저장 ===

# 가장 좋은 결과 저장
all_configs = [
    ('K5', metrics_k5, all_results[THRESHOLD_FIXED]),
    ('K10', metrics_k10, combined_k10),
    ('K10_domain', metrics_domain, combined_domain),
]

best_name, best_m, best_df = max(all_configs, key=lambda x: x[1]['overall_acc_strict'])
print(f'최적: {best_name} (Strict={best_m["overall_acc_strict"]*100:.2f}%)')

# 타입 통일
for col in best_df.columns:
    if best_df[col].dtype == object:
        best_df[col] = best_df[col].astype(str)
best_df['is_null'] = best_df['is_null'].astype(bool)

output_path = PATH_CONFIG['processed'] / 'loop_results.parquet'
best_df.to_parquet(output_path, index=False)
print(f'저장: {output_path}')

# 메트릭 저장
eval_data = {
    'best_config': best_name,
    'K5_t0.3': metrics_k5,
    'K10_t0.3': metrics_k10,
    'K10_domain_t0.3': metrics_domain,
    'threshold_comparison': all_metrics,
}
eval_path = PATH_CONFIG['processed'] / 'loop_evaluation.json'
with open(eval_path, 'w', encoding='utf-8') as f:
    json.dump(eval_data, f, ensure_ascii=False, indent=2, default=str)
print(f'메트릭 저장: {eval_path}')

# 비용
total_llm = 9680 + len(low_k10) + len(low_domain)
est_cost = total_llm * 1100 / 1e6 * 0.10 + total_llm * 50 / 1e6 * 0.40
cost_krw = est_cost * 1450
total_spent = 5200 + 1000 + 160 + cost_krw
print(f'\n누적: ~₩{total_spent:,.0f} / ₩50,000, 잔여: ~₩{50000-total_spent:,.0f}')

최적: K10_domain (Strict=52.61%)
저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/loop_results.parquet
메트릭 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/loop_evaluation.json

누적: ~₩9,205 / ₩50,000, 잔여: ~₩40,795


## 15. 세션 종료
세션을 자동으로 종료하고, 자원을 반환한다.

In [19]:
from google.colab import runtime
runtime.unassign()